In [1]:
from dataset import load_dataset
import numpy as np
from tqdm import tqdm
import pandas as pd
from score_PHM import get_regression_score
import joblib
import os
from regression.RandomForest.random_forest_regression import RandomForestRegressorModel
from regression.PolynomialRegressor.polynomial_regression import PolynomialRegression
from sklearn.model_selection import train_test_split
from regression.GPR.gpr_model import GPRModel
from regression.probabilistic_rf_scoring import fit_best_pdf

In [4]:
TARGET_COL = "trq_target"
FEATURES=["trq_measured","mgt","ias","oat","np_ng_ratio","pa"]
OUTPUT_PATH_CSV= "../../output/bayesian_ridge/pred_margin_test_bayesian.csv"
SCALER_FILE="../../output/GPR/model_target/scaler_gpr.pkl"
TRAIN_PATH_SPLIT= "../../data/processed/split_85_15/train_85.csv"
TEST_PATH_SPLIT= "../../data/processed/split_85_15/test_15.csv"
OUTPUT_PATH_RANDOM_FOREST= "../../output/random_forest_regressor/model_target/random_forest_model_target.pkl"
OUTPUT_PATH_POLINOMIAL= "../../output/polynomial_regression/model_target/polynomial_regression_model_target.pkl"
OUTPUT_PATH_GPR_TARGET = "../../output/GPR/model_target/gpr_model_5000.pkl"
OUTPUT_PATH_RESIDUI="../../output/polynomial_regression/model_target/residui.csv"
OUTPUT_PATH_PR_TEST_10= "../../output/polynomial_regression/model_target/test40.csv"
OUTPUT_PATH_RM_TEST_10="../../output/random_forest_regressor/model_target/test40.csv"
OUTPUT_PATH_GPR_TEST_10="../../output/GPR/margin/test40.csv"
OUTPUT_PATH_GPR_TARGET_TEST_10="../../output/GPR/model_target/test10.csv"
ITERATIONS=1000
FEUTURE=["trq_measured","mgt","ias","oat","np_ng_ratio","pa"]

In [9]:
train_df = load_dataset(TRAIN_PATH_SPLIT)
test_df = load_dataset(TEST_PATH_SPLIT)
X_train_60, X_test_40 = train_test_split(
    test_df,
    test_size=0.4,
    random_state=42,
    shuffle=True
)
X_test = test_df[FEUTURE].values
y_test = test_df[TARGET_COL].values

In [5]:
rf_model = RandomForestRegressorModel(
    n_estimators=500,
    min_samples_leaf=5
)
if os.path.exists(OUTPUT_PATH_RANDOM_FOREST):
    rf_model=joblib.load(OUTPUT_PATH_RANDOM_FOREST)
    print("Caricato")

Caricato


In [6]:
poly_model = PolynomialRegression(degree=2)
if os.path.exists(OUTPUT_PATH_POLINOMIAL):
    poly_model= joblib.load(OUTPUT_PATH_POLINOMIAL)
    print("Caricato")

Caricato


## Polinomyal Regressor


In [10]:
if not os.path.exists(OUTPUT_PATH_PR_TEST_10):
    residui=load_dataset(OUTPUT_PATH_RESIDUI)
    residual_arr = residui["residual"].astype(float).to_numpy()
    rng = np.random.default_rng(42)

    rows_out = []

    pbar = tqdm(range(len(X_test_40)), desc="Processing test set", unit="row")

    for i in pbar:

        test_row = X_test_40.iloc[i]
        trq_measured = float(test_row["trq_measured"])
        trq_target_true = float(test_row["trq_target"])
        trq_margin_true = float(test_row["trq_margin"])

        test_x = test_row[FEATURES].to_frame().T

        pbar.set_postfix_str("predicting")
        trq_target_pred = float(poly_model.predict(test_x)[0])
        trq_margin_pred = (trq_measured / trq_target_pred - 1.0) * 100.0

        pbar.set_postfix_str("sampling residuals")
        sampled_res = rng.choice(residual_arr, size=int(ITERATIONS), replace=True)
        trq_target_prob = trq_target_pred + sampled_res

        trq_target_prob = np.where(np.abs(trq_target_prob) < 1e-12, np.nan, trq_target_prob)

        trq_margin_dist = (trq_measured / trq_target_prob - 1.0) * 100.0
        trq_margin_dist = trq_margin_dist[np.isfinite(trq_margin_dist)]


        pbar.set_postfix_str("fitting pdf")
        best,results = fit_best_pdf(trq_margin_dist)

        if best is None or best.get("pdf_args") is None:
            continue

        best_pdf = best["pdf_type"]
        best_args = best["pdf_args"]

        pbar.set_postfix_str("scoring")
        score_true = float(get_regression_score(best_pdf, best_args, float(trq_margin_true)))
        score_pred = float(get_regression_score(best_pdf, best_args, float(trq_margin_pred)))

        out = test_row[FEATURES].to_dict()
        out.update({
            "idx": i,
            "trq_target_true": trq_target_true,
            "trq_target_pred": trq_target_pred,
            "trq_margin_true": float(trq_margin_true),
            "trq_margin_pred": float(trq_margin_pred),
            "best_pdf": best_pdf,
            "best_pdf_args": best_args,
            "score_true": score_true,
            "score_pred": score_pred,
        })
        rows_out.append(out)
    pbar.close()
    PL_test_10 = pd.DataFrame(rows_out)
    PL_test_10.to_csv(OUTPUT_PATH_PR_TEST_10)
else:
    print("Test gia presente")
    PL_test_10=load_dataset(OUTPUT_PATH_PR_TEST_10)

Test gia presente


In [11]:
score_mean=np.mean(PL_test_10["score_true"])
print(score_mean)

0.5309820474404597


In [16]:
score_model= 1 - np.abs(PL_test_10["trq_margin_pred"]-PL_test_10["trq_margin_true"])
print(np.mean(score_model))

0.9719743807802025


## Random Forest Regressor

In [17]:
if not os.path.exists(OUTPUT_PATH_RM_TEST_10):
    rows_out = []
    pbar = tqdm(range(len(X_test_40)), desc="Processing test set", unit="row")

    for i in pbar:
        test_row = X_test_40.iloc[i]

        trq_measured = float(test_row["trq_measured"])
        trq_target_true = float(test_row["trq_target"])
        trq_margin_true = float(test_row["trq_margin"])

        # 1) features del campione (1, n_features)
        x0 = test_row[FEATURES].to_numpy().reshape(1, -1)

        # 2) predizioni dei singoli alberi come DF (n_trees righe, 1 colonna)
        pbar.set_postfix_str("predict trees")
        tree_preds = rf_model.predict_trees(x0)  # -> DataFrame con 'trq_target_prediction'

        # 3) torque margin per ogni albero (distribuzione)
        tree_preds["trq_margin_pred"] = (trq_measured / tree_preds["trq_target_prediction"] - 1.0) * 100.0

        # distribuzione (array) e media
        trq_margin_dist = tree_preds["trq_margin_pred"].to_numpy(dtype=np.float64)
        trq_margin_pred = float(trq_margin_dist.mean())

        # target pred medio (media alberi)
        trq_target_pred = float(tree_preds["trq_target_prediction"].to_numpy(dtype=np.float64).mean())

        # 4) fit pdf sulla distribuzione (NON sulla media)
        pbar.set_postfix_str("fitting pdf")
        best,results = fit_best_pdf(trq_margin_dist)

        if best is None or best.get("pdf_args") is None:
            continue

        best_pdf = best["pdf_type"]
        best_args = best["pdf_args"]

        # 5) scoring
        pbar.set_postfix_str("scoring")
        score_true = float(get_regression_score(best_pdf, best_args, trq_margin_true))

        # 6) output row
        out = test_row[FEATURES].to_dict()
        out.update({
            "idx": i,
            "trq_target_true": trq_target_true,
            "trq_target_pred": trq_target_pred,
            "trq_margin_true": trq_margin_true,
            "trq_margin_pred": trq_margin_pred,
            "best_pdf": best_pdf,
            "best_pdf_args": best_args,
            "score_true": score_true,
        })

        rows_out.append(out)

    pbar.close()

    RM_test_10 = pd.DataFrame(rows_out)
    RM_test_10.head()
    RM_test_10.to_csv(OUTPUT_PATH_RM_TEST_10)
else:
    RM_test_10 = load_dataset(OUTPUT_PATH_RM_TEST_10)


In [18]:
score_mean_RM=np.mean(RM_test_10["score_true"])
print(score_mean_RM)

0.684195215681947


In [19]:
score_model= 1 - np.abs(RM_test_10["trq_margin_pred"]-RM_test_10["trq_margin_true"])
print(np.mean(score_model))

0.7923612992044672


## GPR TARGET

In [23]:
scaler = joblib.load(SCALER_FILE)
test_X=scaler.fit_transform(test_df[FEATURES])
test_y_target=test_df[TARGET_COL]

In [24]:
gpr_model_target = GPRModel(random_state=42)
if os.path.exists(OUTPUT_PATH_GPR_TARGET):
    print("Modello pre-addestrato trovato! Caricamento in corso...")
    try:
        gpr_model_target = joblib.load(OUTPUT_PATH_GPR_TARGET)
        print("SUCCESSO: Modello caricato correttamente.")
        addestramento_necessario = False
    except Exception as e:
        print(f"Errore nel caricamento dei file (file corrotti?): {e}")
        addestramento_necessario = True
else:
    print("Modello non trovato o incompleto. Avvio procedura di addestramento...")
    addestramento_necessario = True

Modello pre-addestrato trovato! Caricamento in corso...
SUCCESSO: Modello caricato correttamente.


In [26]:
if not os.path.exists(OUTPUT_PATH_GPR_TARGET_TEST_10):
    rows_out = []
    pbar = tqdm(range(len(test_X)), desc="Processing test set", unit="row")

    for i in pbar :
        test_row = test_X[i:i+1]
        trq_measured_real = float(test_df.iloc[i]['trq_measured'])
        trq_margin_true = float(test_df.iloc[i]['trq_margin'])
        mu,std = gpr_model_target.predict(test_row)
        mu_target=float(mu[0])
        std_target=float(std[0])
        mu_margin = 100 * (trq_measured_real / mu_target - 1)
        # 3. propagazione dell'incertezza con il Metodo Delta
        std_margin = np.abs(-100 * trq_measured_real / (mu_target**2)) * std_target
        pdf_args = {"loc": mu_margin, "scale": std_margin}

        pbar.set_postfix_str("scoring")
        score_true = float(get_regression_score("norm", pdf_args, trq_margin_true))

        out={
            "idx": i,
            "trq_margin_true": trq_margin_true,
            "trq_margin_pred": mu_margin,
            "pdf": "norm",
            "pdf_args": pdf_args,
            "score_true": score_true,
        }
        rows_out.append(out)

    pbar.close()

    GPR_target_test_10 = pd.DataFrame(rows_out)
    GPR_target_test_10.to_csv(OUTPUT_PATH_GPR_TARGET_TEST_10)
else:
    GPR_target_test_10 = load_dataset(OUTPUT_PATH_GPR_TARGET_TEST_10)

Processing test set: 100%|██████████| 111394/111394 [42:01<00:00, 44.17row/s, scoring]  


In [27]:
score_mean_GPR_target=np.mean(GPR_target_test_10["score_true"])
print(score_mean_GPR_target)

0.657137793979394


In [28]:
score_model= 1 - np.abs(GPR_target_test_10["trq_margin_pred"]-GPR_target_test_10["trq_margin_true"])
print(np.mean(score_model))

0.9165769281517273


## Bayesian Ridge

In [9]:
BR_taget_test= load_dataset(OUTPUT_PATH_CSV)

In [12]:
BR_taget_test["score_true"] = BR_taget_test.apply(
    lambda row: get_regression_score(
        pdf_args={
            "loc": row["trq_margin_pred"],
            "scale": row["std_margin_pred"]
        },
        pdf_type="norm",
        true_target=row["trq_margin_true"]
    ),
    axis=1
)

In [13]:
score_mean_BR_target=np.mean(BR_taget_test["score_true"])
print(score_mean_BR_target)

0.828910750141075


In [8]:
score_model_BR= 1 - np.abs(BR_taget_test["trq_margin_pred"]-BR_taget_test["trq_margin_true"])
print(np.mean(score_model_BR))

0.9650093775451126


In [14]:
BR_taget_test.to_csv(OUTPUT_PATH_CSV)